# PDF → Markdown 変換ノートブック（vol.2 Google Colab版）

Google Drive上の `pdf-to-markdown/input/` にあるPDFを変換し、`output/` にMarkdownを生成します。

## 使い方
1. **セル1** を実行してセットアップ（初回・ランタイム再起動後のみ）
2. **セル2** を実行してGoogle Driveをマウント
3. **セル3** を実行して変換処理を実行
4. **セル4** で出力ファイルを確認

またはメニューから「ランタイム」→「すべてのセルを実行」で一括実行。

---

**フォルダ構成（Googleドライブ）**
```
マイドライブ/
└── pdf-to-markdown/
    ├── input/    ← PDFをここにアップロード
    ├── output/   ← 変換済みMarkdownが生成される
    ├── done/     ← 処理済みPDFが移動される
    └── logs/     ← 処理ログ
```

In [ ]:
# セル1: セットアップ（初回・ランタイム再起動後に実行）
import subprocess
import threading
import time

# ① Java インストール
print("Java をインストール中...")
subprocess.run(["apt-get", "install", "-y", "-q", "default-jdk"], check=True)

# ② opendataloader-pdf インストール
print("opendataloader-pdf をインストール中...")
subprocess.run(
    ["pip", "install", "opendataloader-pdf[hybrid]", "-q"],
    check=True,
)

# ③ hybridバックエンド起動（OCR用）
print("hybridバックエンドを起動中...")
_hybrid_proc = None

def _start_hybrid():
    global _hybrid_proc
    _hybrid_proc = subprocess.Popen([
        "opendataloader-pdf-hybrid",
        "--port", "5002",
        "--ocr-lang", "ja,en",
    ])

t = threading.Thread(target=_start_hybrid, daemon=True)
t.start()
time.sleep(10)  # バックエンド起動待機
print("✅ セットアップ完了")

In [ ]:
# セル2: Googleドライブのマウントとフォルダ設定
import os
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

# ---- フォルダパス設定 ----
# Drive上のフォルダ名を変更した場合はここを書き換えてください
BASE_DIR   = Path("/content/drive/MyDrive/pdf-to-markdown")
INPUT_DIR  = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "output"
DONE_DIR   = BASE_DIR / "done"
LOG_DIR    = BASE_DIR / "logs"

for d in [INPUT_DIR, OUTPUT_DIR, DONE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✅ Driveマウント完了")
print(f"   BASE_DIR: {BASE_DIR}")

In [ ]:
# セル3: 変換実行
import logging
import shutil
import tempfile
from datetime import datetime

import opendataloader_pdf

# ---- 設定 ----
HYBRID_MODE    = "docling-fast"  # スキャンPDF不要なら None に変更
USE_STRUCT_TREE = True

# ---- ログ設定 ----
_log_file = LOG_DIR / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.FileHandler(_log_file, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger("pdf_to_markdown")

# ---- ヘルパー ----
def _timestamp() -> str:
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def _build_kwargs(input_paths: list[str], output_dir: Path) -> dict:
    kwargs = {
        "input_path": input_paths,
        "output_dir": str(output_dir),
        "format": "markdown",
        "use_struct_tree": USE_STRUCT_TREE,
    }
    if HYBRID_MODE:
        kwargs["hybrid"] = HYBRID_MODE
    return kwargs

def _convert_single(pdf: Path) -> Path:
    """単体PDFを変換し output/{stem}.md を返す。"""
    opendataloader_pdf.convert(**_build_kwargs([str(pdf)], OUTPUT_DIR))
    result = OUTPUT_DIR / f"{pdf.stem}.md"
    if not result.exists():
        raise FileNotFoundError(f"変換結果が生成されませんでした: {result}")
    logger.info("変換: %s → %s", pdf.name, result.name)
    return result

def _convert_folder(folder: Path) -> Path:
    """フォルダ内のPDFをファイル名昇順で変換し、単一Markdownへ連結する。"""
    pdfs = sorted(folder.glob("*.pdf"), key=lambda p: p.name)
    if not pdfs:
        raise ValueError(f"変換対象のPDFが空です: {folder.name}")

    with tempfile.TemporaryDirectory() as tmp:
        tmp_dir = Path(tmp)
        opendataloader_pdf.convert(**_build_kwargs([str(p) for p in pdfs], tmp_dir))

        parts: list[str] = []
        missing: list[str] = []
        for pdf in pdfs:
            md = tmp_dir / f"{pdf.stem}.md"
            if md.exists():
                parts.append(md.read_text(encoding="utf-8"))
            else:
                missing.append(pdf.name)
                logger.warning("変換結果が見つかりません: %s", md.name)

        if missing:
            raise FileNotFoundError(
                f"変換結果が不足しているため連結Markdownを生成できません: "
                f"{folder.name} ({len(missing)}件欠落: {', '.join(missing)})"
            )

        final = OUTPUT_DIR / f"{folder.name}.md"
        final.write_text("\n\n".join(parts), encoding="utf-8")

    logger.info("変換: %s/ (%d件) → %s", folder.name, len(pdfs), final.name)
    return final

def _move_to_done(src: Path) -> Path:
    """ファイル/ディレクトリをタイムスタンプ付きで done/ へ移動する。"""
    stamp = _timestamp()
    dst = DONE_DIR / (f"{src.stem}_{stamp}{src.suffix}" if src.is_file() else f"{src.name}_{stamp}")
    shutil.move(str(src), str(dst))
    logger.info("移動: %s → done/%s", src.name, dst.name)
    return dst

# ---- 実行 ----
results = {"success": [], "error": []}

items = sorted(INPUT_DIR.iterdir(), key=lambda p: p.name)
if not items:
    print("⚠️  input/ にファイルがありません")
else:
    for item in items:
        try:
            if item.is_file() and item.suffix.lower() == ".pdf":
                _convert_single(item)
                _move_to_done(item)
                results["success"].append(item.name)
            elif item.is_dir():
                _convert_folder(item)
                _move_to_done(item)
                results["success"].append(item.name + "/")
        except Exception as e:
            logger.error("%s: %s", item.name, e)
            results["error"].append(f"{item.name}: {e}")

print(f"\n✅ 成功: {len(results['success'])}件")
for name in results["success"]:
    print(f"   - {name}")
if results["error"]:
    print(f"❌ 失敗: {len(results['error'])}件")
    for msg in results["error"]:
        print(f"   - {msg}")
print(f"\nログ: {_log_file}")

In [ ]:
# セル4: 出力ファイル一覧確認
md_files = sorted(OUTPUT_DIR.glob("*.md"), key=lambda p: p.name)

if not md_files:
    print("output/ にMarkdownファイルがありません")
else:
    print(f"output/ のMarkdownファイル ({len(md_files)}件):")
    for f in md_files:
        size_kb = f.stat().st_size // 1024
        print(f"  📄 {f.name}  ({size_kb} KB)")